# Train Audio Librosa Models

Run this notebook for the audio-only modeling task.

Inputs:
- `data/audio/master_audio_features.csv`

Outputs saved to Google Drive under `/content/drive/MyDrive/CLARIFY`:
- `CLARIFY/audio/model_outputs/audio_recommender_index.joblib`
- `CLARIFY/audio/model_outputs/audio_hit_score_models.joblib` if a hit-score or rank target exists
- `CLARIFY/audio/model_outputs/audio_model_metadata.json`

What it does:
- mounts Google Drive
- checks the Librosa feature columns
- trains optional audio-only hit-score baselines
- builds an audio-only similarity index
- avoids Spotify IDs and Spotify API features entirely

## 1. Setup

Installs/imports the basic data science libraries used in this notebook.

In [ ]:
!pip install -q pandas numpy scikit-learn joblib

In [ ]:
from pathlib import Path
import json
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

## 2. Paths

Finds the input CSV from the repo or Drive, then saves all model artifacts to `/content/drive/MyDrive/CLARIFY`.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/CLARIFY')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    DRIVE_ROOT,
    Path('/content/DS3-CLARIFY'),
    Path('/content/drive/MyDrive/DS3-CLARIFY'),
]
PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / 'data' / 'audio' / 'master_audio_features.csv').exists()),
    DRIVE_ROOT,
)

AUDIO_DATA_DIR = PROJECT_ROOT / 'data' / 'audio'
MODEL_DIR = DRIVE_ROOT / 'audio' / 'model_outputs'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MASTER_AUDIO_PATH = AUDIO_DATA_DIR / 'master_audio_features.csv'
AUDIO_RECOMMENDER_PATH = MODEL_DIR / 'audio_recommender_index.joblib'
AUDIO_HIT_MODELS_PATH = MODEL_DIR / 'audio_hit_score_models.joblib'
AUDIO_METADATA_PATH = MODEL_DIR / 'audio_model_metadata.json'

TARGET_COLUMN = 'HitScore'
RANK_COLUMNS = ['Billboard_Rank', 'Rank', 'Peak_Rank', 'peak_rank']

print('Project root:', PROJECT_ROOT)
print('Drive artifact root:', DRIVE_ROOT)
print('Master audio:', MASTER_AUDIO_PATH, MASTER_AUDIO_PATH.exists())
print('Audio artifacts save to:', MODEL_DIR)

## 3. Load Librosa Features

Loads the master audio table and verifies that tempo, MFCC, chroma, and spectral centroid columns exist.

In [ ]:
audio_df = pd.read_csv(MASTER_AUDIO_PATH)

IDENTITY_COLUMNS = ['SONG_ID', 'SONG_TITLE', 'ARTIST_NAME']
LIBROSA_FEATURE_COLUMNS = (
    ['tempo']
    + [f'mfcc_{i}' for i in range(1, 14)]
    + [f'chroma_mean_{i}' for i in range(1, 13)]
    + [f'chroma_std_{i}' for i in range(1, 13)]
    + ['spectral_centroid']
)
OPTIONAL_METADATA_FEATURES = ['year']

missing_identity = [col for col in IDENTITY_COLUMNS if col not in audio_df.columns]
missing_librosa = [col for col in LIBROSA_FEATURE_COLUMNS if col not in audio_df.columns]
if missing_identity:
    raise ValueError(f'Missing identity columns: {missing_identity}')
if missing_librosa:
    raise ValueError(f'Missing expected Librosa columns: {missing_librosa}')

AUDIO_INPUT_COLUMNS = [col for col in OPTIONAL_METADATA_FEATURES + LIBROSA_FEATURE_COLUMNS if col in audio_df.columns]
for col in AUDIO_INPUT_COLUMNS:
    audio_df[col] = pd.to_numeric(audio_df[col], errors='coerce')

print('Rows:', len(audio_df))
print('Columns:', len(audio_df.columns))
print('Audio input columns:', len(AUDIO_INPUT_COLUMNS))
audio_df[IDENTITY_COLUMNS + AUDIO_INPUT_COLUMNS].head()

## 4. Clean Feature Matrix

Builds the numeric audio feature matrix and fills missing feature values with column medians.

In [ ]:
model_audio = audio_df[IDENTITY_COLUMNS + AUDIO_INPUT_COLUMNS].copy()
feature_medians = model_audio[AUDIO_INPUT_COLUMNS].median(numeric_only=True)
model_audio[AUDIO_INPUT_COLUMNS] = model_audio[AUDIO_INPUT_COLUMNS].fillna(feature_medians)

print('Rows after cleaning:', len(model_audio))
print('Remaining missing values:', int(model_audio[AUDIO_INPUT_COLUMNS].isna().sum().sum()))

## 5. Optional Audio Hit-Score Model

Runs only if a real target exists. Otherwise, this section prints a skip message and the recommender still runs.

In [ ]:
def attach_target(df):
    df = df.copy()
    if TARGET_COLUMN in df.columns:
        df[TARGET_COLUMN] = pd.to_numeric(df[TARGET_COLUMN], errors='coerce')
        return df, TARGET_COLUMN
    rank_column = next((col for col in RANK_COLUMNS if col in df.columns), None)
    if rank_column is not None:
        rank = pd.to_numeric(df[rank_column], errors='coerce')
        df[TARGET_COLUMN] = (101 - rank).clip(lower=0, upper=100) / 100.0
        print(f'Derived {TARGET_COLUMN} from {rank_column}.')
        return df, TARGET_COLUMN
    return df, None

target_audio, target_column = attach_target(audio_df)
hit_score_metrics = None
hit_score_models = {}

if target_column is None:
    print('No hit-score/rank target in master_audio_features.csv. Skipping supervised audio training.')
else:
    supervised = model_audio.merge(target_audio[['SONG_ID', target_column]], on='SONG_ID', how='inner')
    supervised = supervised.dropna(subset=[target_column]).reset_index(drop=True)
    if len(supervised) < 20:
        print(f'Only {len(supervised)} target rows. Skipping until more labels exist.')
    else:
        X = supervised[AUDIO_INPUT_COLUMNS].astype(float)
        y = supervised[target_column].astype(float)
        candidates = {
            'ridge': Pipeline([('scaler', StandardScaler()), ('regressor', Ridge(alpha=10.0))]),
            'random_forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=SEED, n_jobs=-1),
        }
        rows = []
        for name, model in candidates.items():
            cv = KFold(n_splits=min(5, len(supervised)), shuffle=True, random_state=SEED)
            pred = np.zeros(len(supervised))
            for train_idx, test_idx in cv.split(X):
                model.fit(X.iloc[train_idx], y.iloc[train_idx])
                pred[test_idx] = model.predict(X.iloc[test_idx])
            rows.append({
                'Model': name,
                'Rows': len(supervised),
                'Features': len(AUDIO_INPUT_COLUMNS),
                'Target': target_column,
                'MAE': mean_absolute_error(y, pred),
                'RMSE': mean_squared_error(y, pred) ** 0.5,
                'R2': r2_score(y, pred),
            })
            model.fit(X, y)
            hit_score_models[name] = {'model': model, 'feature_columns': AUDIO_INPUT_COLUMNS, 'target_column': target_column}
        hit_score_metrics = pd.DataFrame(rows)
        joblib.dump(hit_score_models, AUDIO_HIT_MODELS_PATH)
        display(hit_score_metrics.round(4))

## 6. Audio Similarity Index

Scales the audio features and builds a cosine-nearest-neighbor recommender over audio only.

In [ ]:
audio_scaler = StandardScaler()
audio_matrix = audio_scaler.fit_transform(model_audio[AUDIO_INPUT_COLUMNS].astype(float).to_numpy())

recommender = NearestNeighbors(metric='cosine', algorithm='brute')
recommender.fit(audio_matrix)

payload = {
    'model': recommender,
    'vectors': audio_matrix,
    'song_metadata': model_audio[IDENTITY_COLUMNS].reset_index(drop=True),
    'feature_columns': AUDIO_INPUT_COLUMNS,
    'scaler': audio_scaler,
}
joblib.dump(payload, AUDIO_RECOMMENDER_PATH)
print('Saved audio recommender:', AUDIO_RECOMMENDER_PATH)

def recommend_audio_neighbors(row_index, k=10):
    distances, indices = recommender.kneighbors(audio_matrix[[row_index]], n_neighbors=min(k + 1, len(model_audio)))
    rows = []
    for distance, idx in zip(distances[0], indices[0]):
        if idx == row_index:
            continue
        rows.append({
            'SONG_TITLE': model_audio.loc[idx, 'SONG_TITLE'],
            'ARTIST_NAME': model_audio.loc[idx, 'ARTIST_NAME'],
            'Similarity': 1 - distance,
        })
    return pd.DataFrame(rows)

recommend_audio_neighbors(0, k=10)

## 7. Save Metadata

Writes a small JSON summary so the team can see what data and artifacts this run produced.

In [ ]:
metadata = {
    'source_csv': str(MASTER_AUDIO_PATH),
    'rows': int(len(model_audio)),
    'input_columns': AUDIO_INPUT_COLUMNS,
    'input_column_count': len(AUDIO_INPUT_COLUMNS),
    'uses_spotify_api': False,
    'uses_spotify_ids': False,
    'target_column': target_column,
    'artifacts': {
        'audio_recommender': str(AUDIO_RECOMMENDER_PATH),
        'audio_hit_models': str(AUDIO_HIT_MODELS_PATH) if hit_score_models else None,
    },
}
AUDIO_METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
metadata